In [ ]:
from dotenv import load_dotenv
import pandas as pd
import numpy as np
import os
import json
from cmath import pi
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import root_mean_squared_error, r2_score, mean_absolute_error, mean_absolute_percentage_error
from scipy.stats import ttest_rel

In [ ]:
load_dotenv(override=True)

basic_folder_path = os.getenv("BASIC_FOLDER")

expert_df = pd.read_excel(os.getenv("EXPERT_CLEANED_FILE"))
anthropic_df = pd.read_excel(basic_folder_path+'/nutria_anthropic.xlsx')
gemini_df = pd.read_excel(basic_folder_path+'/nutria_gemini.xlsx')
gpt_df = pd.read_excel(basic_folder_path+'/nutria_gpt.xlsx')

In [ ]:
expert_df.columns, anthropic_df.columns, gemini_df.columns, gpt_df.columns

In [ ]:
expert_df = expert_df.rename(columns={'description': 'expert_description', 'serving_size': 'expert_serving_size', 'calories': 'expert_calories', 'proteins': 'expert_proteins', 'carbohydrates': 'expert_carbohydrates', 'fats': 'expert_fats', 'observaciones': 'observations'})
anthropic_df = anthropic_df.rename(columns={'description': 'anthropic_description', 'serving_size': 'anthropic_serving_size', 'calories': 'anthropic_calories', 'proteins': 'anthropic_proteins', 'carbohydrates': 'anthropic_carbohydrates', 'fats': 'anthropic_fats'})
gemini_df = gemini_df.rename(columns={'description': 'gemini_description', 'serving_size': 'gemini_serving_size', 'calories': 'gemini_calories', 'proteins': 'gemini_proteins', 'carbohydrates': 'gemini_carbohydrates', 'fats': 'gemini_fats'})
gpt_df = gpt_df.rename(columns={'description': 'gpt_description', 'serving_size': 'gpt_serving_size', 'calories': 'gpt_calories', 'proteins': 'gpt_proteins', 'carbohydrates': 'gpt_carbohydrates', 'fats': 'gpt_fats'})

In [ ]:
df = pd.merge(expert_df, gemini_df, on='id')
df = pd.merge(df, gpt_df, on='id')
df = pd.merge(df, anthropic_df, on='id')
df.shape

In [ ]:
df.head(5)

In [ ]:
df.describe()

In [ ]:
df = df[df['expert_serving_size']!=0]
df = df[df['id']!=31]
df = df[df['id']!=28]
df.shape

In [ ]:
df.describe()

In [ ]:
def evaluate_models(df, metrics, true_suffix='expert_', pred_suffixes=['gemini_', 'gpt_', 'anthropic_']):
    all_results = []

    for pred_suffix in pred_suffixes:
        for metric in metrics:
            y_true = df[f"{true_suffix}{metric}"]
            y_pred = df[f"{pred_suffix}{metric}"]

            mae = mean_absolute_error(y_true, y_pred)
            mask = y_true > 1e-3
            mape = mean_absolute_percentage_error(y_true[mask], y_pred[mask]) * 100 if mask.sum() > 0 else np.nan
            rmse = root_mean_squared_error(y_true, y_pred)
            r2 = r2_score(y_true, y_pred)
            within_20pct = np.mean(np.abs(y_true - y_pred) / y_true <= 0.20) * 100
            t_stat, p_value = ttest_rel(y_true, y_pred)

            all_results.append({
                "Model": pred_suffix.strip('_').upper(),
                "Metric": metric.capitalize(),
                "MAE": round(mae, 2),
                "MAPE (%)": round(mape, 2) if not np.isnan(mape) else "N/A",
                "RMSE": round(rmse, 2),
                "R2 Score": round(r2, 3),
                "Within ±20%": round(within_20pct, 2),
                "T-test p-value": round(p_value, 4)
            })

    return pd.DataFrame(all_results)

In [ ]:
def rank_models(result_df):
    higher_better = {
        "MAE": False,
        "MAPE (%)": False,
        "RMSE": False,
        "R2 Score": True,
        "Within ±20%": True
    }

    score_df = result_df.copy()

    for metric, high in higher_better.items():
        vals = pd.to_numeric(score_df[metric], errors='coerce')
        norm = (vals - vals.min()) / (vals.max() - vals.min())
        if not high:
            norm = 1 - norm
        score_df[f"{metric}_score"] = norm

    score_columns = [col for col in score_df.columns if col.endswith('_score')]
    model_avg_scores = score_df.groupby("Model")[score_columns].mean()
    model_avg_scores["Average Score"] = model_avg_scores.mean(axis=1)
    model_avg_scores["Rank"] = model_avg_scores["Average Score"].rank(ascending=False)

    return model_avg_scores.sort_values("Rank").reset_index()


In [ ]:
def plot_heatmap(df, metric_cols):
    df_norm = df.copy()
    df_norm[metric_cols] = df_norm[metric_cols].apply(lambda x: (x - x.min()) / (x.max() - x.min()))

    for metric in metric_cols:
        plt.figure(figsize=(6, 4))
        sns.heatmap(
            df_norm.pivot(index="Metric", columns="Model", values=metric),
            annot=True,
            cmap="RdYlGn",  # red-yellow-green colormap
            cbar=True,
            fmt=".2f",
            linewidths=0.5,
            linecolor='gray'
        )
        plt.title(f"Heatmap of {metric} (Normalized)")
        plt.tight_layout()
        plt.show()


In [ ]:
def plot_bar_comparison(df, metric_name):
    plt.figure(figsize=(8, 5))
    sns.barplot(data=df, x="Metric", y=metric_name, hue="Model")
    plt.title(f"{metric_name} Comparison Across Models")
    plt.ylabel(metric_name)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

In [ ]:
def plot_radar_per_metric(df, metrics):
    categories = list(df["Metric"].unique())

    for metric in metrics:
        values_dict = {}
        for model in df["Model"].unique():
            values = df[df["Model"] == model][metric].astype(float).values
            values = (values - np.min(values)) / (np.max(values) - np.min(values))  # Normalize
            values_dict[model] = np.concatenate((values, [values[0]]))  # Close the radar chart

        angles = [n / float(len(categories)) * 2 * pi for n in range(len(categories))]
        angles += angles[:1]

        fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))
        for model, values in values_dict.items():
            ax.plot(angles, values, label=model)
            ax.fill(angles, values, alpha=0.1)

        ax.set_xticks(angles[:-1])
        ax.set_xticklabels(categories)
        ax.set_title(f"Radar Chart for {metric} (Normalized)")
        ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
        plt.tight_layout()
        plt.show()


In [ ]:
metrics = ['serving_size', 'calories', 'proteins', 'carbohydrates', 'fats']
all_results = []

# true results vs predictions for each model
true_suffix = 'expert_'
pred_suffixes = ['gemini_', 'gpt_', 'anthropic_']

In [ ]:
results_df = evaluate_models(df, metrics)
ranking_df = rank_models(results_df)

# Visualize
metric_columns = ["MAE", "MAPE (%)", "RMSE", "R2 Score", "Within ±20%"]
plot_heatmap(results_df, metric_columns)

for metric in metric_columns:
    plot_bar_comparison(results_df, metric)

plot_radar_per_metric(results_df, metric_columns)

# Show final ranking
ranking_df
